# Notebook 04 -- reference pools

**Asks:** build the reference passages the embeddings compare against.

The embedding leg of the measurement scores a response by asking how close it sits, in
meaning-space, to a small set of passages that are known to express a facet. This
notebook builds those sets -- eight to fifteen short passages per facet -- and writes
them, with their provenance, to `embedding_seed_pools.json`.

Two terms are used throughout and are worth pinning down once. A response's facet score
is its **mean cosine** to the facet's set: cosine is the standard closeness measure for
embeddings, running from 1.0 for two passages pointing the same way in meaning-space to
0.0 for two that share no direction at all, and the score is the average of a response's
cosine to each passage in the set. The **centroid** of a set is the average position of
its passages, the single point in meaning-space they cluster around, and it is what a
mean cosine measures distance from. Both are why the sets need eight to fifteen members:
a centroid resting on two or three passages is one unusual passage away from sitting
somewhere else entirely.

Two rules govern the build, and both are about not marking our own homework:

1. **Nothing from the TEST half.** The frozen SEED/TEST split (notebook 03) exists so
   that every convergence number is computed on responses that contributed nothing to
   the thing being checked. A reference passage drawn from a TEST response cannot fail
   to look similar to itself, and the agreement it earned would be a construction, not
   a finding.
2. **Nothing chosen by the ensemble.** The passages come from two routes only -- the
   codebook's own written anchor passages, and responses the word list scored positive.
   Neither route consults a language model, so the embedding leg stays an independent
   check on the ensemble rather than a reflection of it.

**Where the passages come from, and how that differs from the design's first choice.**
MEASUREMENT_DESIGN.md section 5 seeds these pools from responses that human coders
scored highest on a facet. Prereg 6.7 puts an ensemble of language models in place of
human raters and prescribes no human-coding step, so under the registered pipeline
there are no human codes to seed from, and the two model-free sources that exist
without a coder are used instead. The human-coded arm, when it is switched on, is
reported here as a comparison and never changes the pools.

Every exemplar carries its route, its source, its split label and the random seed into
the JSON, so a reader can audit where each reference passage came from.

| | |
|---|---|
| Reads | `openends_long.csv` (00), `facet_dictionary_scores.csv` (01), `gold_split_assignment.csv` (03), `gold_codes.csv` (99, optional) |
| Writes | `embedding_seed_pools.json` |

## Config

In [1]:
# === CONFIG: the only cell you edit ===
# Every switch in the pipeline lives here. Nothing below this cell needs changing.

RUN_MODE    = "full"        # "smoke" = first 20 rows, costs pennies, proves the
                            #           whole chain works end to end.
                            # "full"  = the real corpus.
HUMAN_GOLD  = False         # False = the pre-registered pipeline. Language models
                            #         do the coding, exactly as prereg 6.7 says.
                            #         This is the default and it runs start to
                            #         finish with no human step.
                            # True  = ALSO run the human-coder arm. That arm is a
                            #         DEVIATION from the registration and needs the
                            #         OSF addendum posted first.
WORKDIR     = "/content/retribution_nlp"   # where every file in the chain lives
RANDOM_SEED = 20260629      # frozen. Changing it re-draws every random split.

# --- PER-NOTEBOOK: the three lines below differ in each notebook ---
NB_ID       = "04"                     # this notebook's number
NB_NAME     = "reference_pools"        # this notebook's job, in two words
NB_ASKS     = "Build the reference passages the embeddings compare against."

## Setup

In [2]:
# === SETUP: folders, the manifest, and the two functions every notebook uses ===
# WHAT THIS CELL DOES: makes the working folder, loads the manifest, and defines
# need() and done(). You never edit this cell.
#
# THE MANIFEST is a small file called _manifest.json that lives beside the data.
# Every notebook records what it produced in there. The next notebook reads it to
# check its inputs exist. That is what lets a notebook tell you "run 03 first"
# instead of dying on a missing file three cells later.

import os, json, hashlib, datetime, sys

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
MANIFEST = os.path.join(WORKDIR, "_manifest.json")

def _load_manifest():
    """Read the manifest, or start an empty one on the first notebook."""
    if os.path.exists(MANIFEST):
        with open(MANIFEST) as f:
            return json.load(f)
    return {"files": {}, "runs": []}

def _sha256(path, cap=8_000_000):
    """First 8 MB is plenty to catch a file that changed underneath us."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(cap))
    return h.hexdigest()[:16]

class MissingInput(Exception):
    """Raised when a required upstream file is not there yet."""
    pass

def need(filename, produced_by=None, optional=False):
    """Return the path to an input file, or explain which notebook makes it.

    This is the function that makes the chain navigable. A missing input stops
    the notebook HERE, with a sentence telling you what to run, rather than
    surfacing as a confusing error further down.
    """
    path = os.path.join(WORKDIR, filename)
    if os.path.exists(path):
        rec = _load_manifest()["files"].get(filename, {})
        stamp = rec.get("written_at", "unknown time")
        print(f"  [ok]   {filename}  (written {stamp})")
        return path
    if optional:
        print(f"  [skip] {filename} is not here. This notebook will carry on "
              f"without it and say so in its output.")
        return None
    where = f" Run notebook {produced_by} first." if produced_by else ""
    raise MissingInput(
        f"\n\n  MISSING INPUT: {filename}\n"
        f"  This notebook cannot run without it.{where}\n"
        f"  Files present in {WORKDIR}:\n    "
        + "\n    ".join(sorted(os.listdir(WORKDIR)) or ["(none)"]) + "\n")

def done(filename, rows=None, note=""):
    """Record a file this notebook produced, so later notebooks can find it."""
    path = os.path.join(WORKDIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"done() called for {filename} but it is not there")
    m = _load_manifest()
    m["files"][filename] = {
        "written_by": f"{NB_ID}_{NB_NAME}",
        "written_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "rows": rows,
        "sha16": _sha256(path),
        "run_mode": RUN_MODE,
        "note": note,
    }
    m["runs"] = [r for r in m["runs"] if r.get("nb") != NB_ID]
    m["runs"].append({"nb": NB_ID, "name": NB_NAME, "run_mode": RUN_MODE,
                      "human_gold": HUMAN_GOLD,
                      "at": datetime.datetime.now().isoformat(timespec="seconds")})
    with open(MANIFEST, "w") as f:
        json.dump(m, f, indent=1)
    size = os.path.getsize(path)
    print(f"  [wrote] {filename}  ({rows if rows is not None else '?'} rows, "
          f"{size:,} bytes)")

print(f"NOTEBOOK {NB_ID} -- {NB_NAME}")
print(f"  {NB_ASKS}")
print(f"  mode: RUN_MODE={RUN_MODE}, HUMAN_GOLD={HUMAN_GOLD}")
print(f"  folder: {WORKDIR}")
_m = _load_manifest()
if _m["runs"]:
    print("  already run in this folder: "
          + ", ".join(sorted(r["nb"] for r in _m["runs"])))
else:
    print("  this folder is empty: notebook 00 is the place to start.")

NOTEBOOK 04 -- reference_pools
  Build the reference passages the embeddings compare against.
  mode: RUN_MODE=full, HUMAN_GOLD=False
  folder: /content/retribution_nlp
  this folder is empty: notebook 00 is the place to start.


## Load

In [3]:
# === LOAD: the corpus, the word-list scores, and the frozen SEED/TEST split ===
# WHAT THIS CELL DOES: fetches every input through need(), so a file that is not
# there yet names the notebook that makes it instead of failing later, halfway
# through the pool builder, as an unreadable pandas error.
# CONSUMES: openends_long.csv           (00) -- the text of every open-end
#           facet_dictionary_scores.csv (01) -- the word-list counts, per facet
#           gold_split_assignment.csv   (03) -- which rows are SEED, which are TEST
#           gold_codes.csv        (99, optional) -- human consensus codes, read only
#                                               when HUMAN_GOLD is True, and used
#                                               only for the comparison cell further
#                                               down. It never enters a pool.
# PRODUCES: df, dict_df, split, gold -- frames keyed on response_id x item_type.
import pandas as pd

print("Inputs:")
P_TEXT  = need("openends_long.csv",           produced_by="00")
P_DICT  = need("facet_dictionary_scores.csv", produced_by="01")
P_SPLIT = need("gold_split_assignment.csv",   produced_by="03")
# The human-coded gold set is OPTIONAL and off by default. The registration (prereg
# 6.7) puts the ensemble in place of human raters and prescribes no human-coding step,
# so the default path never asks for this file; when it is switched on and the file is
# absent, need(optional=True) prints one line and the notebook carries on.
P_GOLD  = (need("gold_codes.csv", produced_by="99", optional=True)
           if HUMAN_GOLD else None)
if not HUMAN_GOLD:
    print("  [skip] gold_codes.csv not requested (HUMAN_GOLD=False). Pools are built "
          "from the codebook anchors and the word list alone.")

# keep_default_na=False, matching the way notebook 00 wrote the file. A participant who
# literally answered "N/A" keeps that text instead of arriving as a missing value --
# and these texts are copied verbatim into the reference pools, so a missing value here
# would be written into the JSON as a null passage and silently blunt a facet centroid.
df      = pd.read_csv(P_TEXT, keep_default_na=False)
dict_df = pd.read_csv(P_DICT)
split   = pd.read_csv(P_SPLIT, keep_default_na=False)
gold    = pd.read_csv(P_GOLD, keep_default_na=False) if P_GOLD else None

# Nothing is touched here beyond reading. The columns are not confirmed to exist yet,
# and a fillna on a column that is not there raises a bare KeyError naming neither the
# file nor the notebook that writes it -- exactly the error this cell exists to avoid.
# The next cell checks the contract and then does the two coercions.
print(f"\n  corpus rows          : {len(df)}")
print(f"  word-list scored rows: {len(dict_df)}")
print("  split assignment     : "
      + ", ".join(f"{k}={v}" for k, v in split["gold_split"].value_counts().items()))
if gold is not None:
    print(f"  human gold rows      : {len(gold)}")

Inputs:
  [ok]   openends_long.csv  (written unknown time)
  [ok]   facet_dictionary_scores.csv  (written unknown time)
  [ok]   gold_split_assignment.csv  (written unknown time)
  [skip] gold_codes.csv not requested (HUMAN_GOLD=False). Pools are built from the codebook anchors and the word list alone.

  corpus rows          : 2844
  word-list scored rows: 2844
  split assignment     : TEST=154, SEED=154


In [4]:
# === THE CONTRACT: one row per response x item, checked here and not assumed ===
# WHAT THIS CELL DOES: checks that each input carries the columns this notebook reads,
# and that no input holds the same (response_id, item_type) pair twice.
# WHY THE UNIQUENESS CHECK IS THE IMPORTANT HALF: every join below keys on that pair,
# and the anchor lookup takes the FIRST matching row. A file holding a pair twice -- a
# re-export appended to an older one, two coder files concatenated, a spreadsheet saved
# with a stray duplicated line -- fans the merge out, doubles a candidate's chance of
# being drawn as an exemplar, and in gold_split_assignment.csv can label one response
# SEED and TEST at the same time, which would quietly disarm the firewall this whole
# notebook is built around. None of that raises anything on its own: the notebook runs
# and writes a plausible file. It is checked HERE, for every file this notebook reads,
# and checked again on the pools themselves before they are written. No other notebook
# is relied on to have done it.
KEYS = ["response_id", "item_type"]

def assert_unique(frame, name, produced_by):
    """Stop if `frame` holds the same response x item pair more than once."""
    dup = frame.duplicated(KEYS, keep=False)
    assert not dup.any(), (
        f"{name} (written by notebook {produced_by}) has {int(dup.sum())} rows "
        f"sharing a response_id x item_type key, for example "
        f"{frame.loc[dup, KEYS].head(3).to_dict('records')}. One row per response per "
        f"item is the contract every join in this chain keys on; fix the file rather "
        f"than dropping duplicates here, because which copy is right is not knowable "
        f"from this notebook.")

# issubset rather than equality, so a file carrying extra columns still passes.
assert {"response_id", "item_type", "text"}.issubset(df.columns), \
    f"openends_long.csv needs response_id, item_type and text; found {sorted(df.columns)}"
assert {"response_id", "item_type", "n_words"}.issubset(dict_df.columns), \
    ("facet_dictionary_scores.csv needs response_id, item_type and n_words; found "
     f"{sorted(dict_df.columns)}")
assert {"response_id", "item_type", "gold_split"}.issubset(split.columns), \
    f"gold_split_assignment.csv is missing columns; found {sorted(split.columns)}"

assert_unique(df,      "openends_long.csv",           "00")
assert_unique(dict_df, "facet_dictionary_scores.csv", "01")
assert_unique(split,   "gold_split_assignment.csv",   "03")
print(f"One row per response x item confirmed in all three required inputs "
      f"({len(df)}, {len(dict_df)} and {len(split)} rows).")

# The two coercions, done here because the columns are only now known to exist. text
# is forced to string because these passages are copied verbatim into the JSON and a
# missing value would be written as a null passage. n_words is coerced on dict_df and
# NOT on df, because dict_df is the frame the MIN_WORDS filter reads: coercing the
# copy that is never read would look like a check and do nothing.
df["text"] = df["text"].fillna("").astype(str)
dict_df["n_words"] = (pd.to_numeric(dict_df["n_words"], errors="coerce")
                        .fillna(0).astype(int))

# gold_codes.csv is optional, so a defect in it is reported and survived rather than
# raised: nothing in the registered pipeline needs it, and a bad optional file must
# never take down the analyses that do not use it. The comparison cell reads this.
gold_problem = ""
if gold is not None:
    if not set(KEYS).issubset(gold.columns):
        gold_problem = f"gold_codes.csv has no {KEYS} columns"
    elif gold.duplicated(KEYS).any():
        gold_problem = (f"gold_codes.csv has {int(gold.duplicated(KEYS).sum())} "
                        f"duplicated response x item rows")
    if gold_problem:
        print(f"  [note] {gold_problem}. The human-coded comparison below is reported "
              f"as not run. Nothing else in this notebook uses that file.")

One row per response x item confirmed in all three required inputs (2844, 2844 and 308 rows).


## Do

In [5]:
# === THE NINE FACETS AND THE THREE CLASSES THEY FALL INTO ===
# WHAT THIS CELL DOES: fixes the facet names, sorts them into the three classes, and
# derives from those classes which facets count as "opposing" when an exemplar is
# picked.
# WHY IT IS DECLARED HERE RATHER THAN IMPORTED: the order fixes the order of every
# printed table and of the pools in the JSON, so the same nine names in the same nine
# places read the same way in every notebook of the chain.
FACET_NAMES = ["desert", "proportionality", "censure", "moral_balance",
               "revenge", "suffering",
               "deterrence", "incapacitation", "rehabilitation"]

# The class groupings. PRINCIPLED and VENGEFUL are the two sides of E4's co-occurrence
# question; CONSEQUENTIALIST is a third axis, scored so the retributive facets are
# measured against a full menu of punishment justifications rather than in a vacuum.
PRINCIPLED       = ["desert", "proportionality", "censure", "moral_balance"]
VENGEFUL         = ["revenge", "suffering"]
CONSEQUENTIALIST = ["deterrence", "incapacitation", "rehabilitation"]

# EVERY FACET SITS IN EXACTLY ONE CLASS, and this line is what makes that true rather
# than intended. A tenth facet added to FACET_NAMES and to neither class list would
# otherwise pick up an empty opposing set by default: its pool would be built with no
# cleanliness ranking at all, and nothing in the output would say so.
_classed = PRINCIPLED + VENGEFUL + CONSEQUENTIALIST
assert sorted(_classed) == sorted(FACET_NAMES) and len(_classed) == len(set(_classed)), \
    ("every facet must appear in exactly one of PRINCIPLED, VENGEFUL and "
     f"CONSEQUENTIALIST. Unclassed: {sorted(set(FACET_NAMES) - set(_classed))}; "
     f"unknown: {sorted(set(_classed) - set(FACET_NAMES))}.")

# Which facets count as "opposing" when picking exemplars, read off the classes above
# rather than typed out a second time, so the two can never drift apart.
OPP_OF = {}
for _f in FACET_NAMES:
    if _f in PRINCIPLED:
        OPP_OF[_f] = VENGEFUL          # the principled facets are opposed by payback
    elif _f in VENGEFUL:
        OPP_OF[_f] = PRINCIPLED        # and vice versa
    else:
        # The consequentialist goals have no opposite here. They are a different axis,
        # not the other end of one, so their pools are picked on facet presence alone.
        OPP_OF[_f] = []

# The word-list columns this notebook reads. A file missing one of them would send
# every candidate query for that facet down a KeyError deep inside the ranking
# function, where the message says nothing about which file is wrong.
_no_col = [f for f in FACET_NAMES if f not in dict_df.columns]
assert not _no_col, (f"facet_dictionary_scores.csv has no column for {_no_col}. "
                     f"Re-run notebook 01; these are the counts the pools are built from.")
print("Nine facets, three classes: "
      f"{len(PRINCIPLED)} principled, {len(VENGEFUL)} vengeful, "
      f"{len(CONSEQUENTIALIST)} consequentialist.")

Nine facets, three classes: 4 principled, 2 vengeful, 3 consequentialist.


In [6]:
# === THE POOL SETTINGS: what a reference pool is allowed to contain ===
POOL_MIN   = 8     # 8 to 15 exemplars per facet is what the design (section 5) sets.
POOL_MAX   = 15    # A pool under POOL_MIN is emptied and reported, never quietly used:
                   # a centroid -- the average position of a set of passages in
                   # meaning-space -- resting on two or three responses is one response
                   # away from meaning something else entirely.
MIXED_MIN  = 2     # slots held for genuine mixed-motive exemplars -- see the pool cell
MAX_CHARS  = 400   # reference passages are SHORT by design; see the pool cell
MIN_WORDS  = 3     # the >=3-word filter used everywhere in this pipeline: a two-word
                   # answer carries no context for an encoder to read
ANCHOR_TARGET_MIN = 0.5   # how strong a codebook target has to be to seed a pool

if RUN_MODE == "smoke":
    # A smoke run looks at 20 participants, so the 8-exemplar floor would empty every
    # pool and leave the notebooks downstream nothing to run on -- which would prove
    # only that this notebook can write a file, not that the chain works end to end.
    # The floor drops to 2 for smoke runs only, and "run_mode": "smoke" is stamped
    # into the JSON so a smoke pool can never be mistaken for a real one.
    POOL_MIN = 2
    print("RUN_MODE=smoke: minimum pool size relaxed to 2 so the chain has something")
    print("to run on. These pools are a plumbing check, not a measurement.")

In [7]:
# === THE CODEBOOK ANCHOR PASSAGES, PART 1: the five clean facet anchors ===
# WHAT THIS CELL DOES: writes down, as data, the anchor passages the codebook
# (MEASUREMENT_DESIGN.md section 4) chose to define each facet, together with the
# facet targets the codebook author assigned them.
# WHY THIS ROUTE EXISTS: these passages were selected and scored by the study authors
# when the codebook was written, before any language model scored anything. A
# reference passage taken from them therefore cannot have been chosen by the ensemble
# whose embedding convergence it is later used to check. That independence is the
# whole value of the route, and it is what lets the pools be built with no human
# coding step at all (prereg 6.7).
# NOTE ON THE QUOTES: the codebook prints them abridged, with "..." where the author
# cut. What is printed here is what the author scored, so it is the passage that
# becomes the reference -- see the route 1 cell for why the full response is not used.
CODEBOOK_ANCHORS = [
    # DESERT (clean). The agent earned it, stated as a backward fact.
    {"response_id": "R_3gSICIiGeQ8DCjN", "item_type": "ranking",
     "label": "DESERT (clean)",
     "quote": ("Criminals deserve to be punished... victims need to feel that "
               "someone pays for the harm done."),
     "targets": {"desert": 0.9, "moral_balance": 0.4}},

    # PROPORTIONALITY (clean). The punishment's calibration -- an amount that fits.
    {"response_id": "R_70vm1xpQlRxHl1s", "item_type": "ranking",
     "label": "PROPORTIONALITY (clean)",
     "quote": ("They should pay a reasonable amount of time away from society to "
               "match their crime."),
     "targets": {"proportionality": 0.9, "incapacitation": 0.3, "desert": 0.2}},

    # CENSURE (proxy). The social message: condemnation, accountability.
    {"response_id": "R_732mg1S5ULj0brc", "item_type": "backward",
     "label": "CENSURE (proxy)",
     "quote": ("The sentencing should always reflect the seriousness of the "
               "crime... should be held accountable."),
     "targets": {"censure": 0.6, "proportionality": 0.5}},

    # REVENGE (clean). Symmetric hitting back, in the participant's own spelling.
    {"response_id": "R_77tp3RXMgnxlR4Z", "item_type": "forward",
     "label": "REVENGE (clean)",
     "quote": "eye for an eye is what i beleave in.",
     "targets": {"revenge": 0.9}},

    # SUFFERING (clean, co-occurs). Pain as the point, here sitting beside a
    # calibration remark -- which is why its proportionality target is nonzero.
    {"response_id": "R_7K3dz3sByxkDqdZ", "item_type": "backward",
     "label": "SUFFERING (clean, co-occurs)",
     "quote": ("since he's paralyzed, he'll suffer more and 40 years seems like "
               "enough time."),
     "targets": {"suffering": 0.7, "proportionality": 0.3}},
]

In [8]:
# === THE CODEBOOK ANCHOR PASSAGES, PART 2: mixed motives, and the stance case ===
# WHAT THIS CELL DOES: adds the two mixed-motive teaching cases, and records the
# hard-negative stance case as a passage that is deliberately NOT used.
# WHY THE MIXED-MOTIVE CASES MATTER: building the reference pools out of pure cases
# only would teach the embedding leg that principled and vengeful language are
# mutually exclusive -- which is precisely the proposition E4 puts under test, so
# assuming it in the reference set would decide the question by construction. A case
# the codebook scores high on both sides therefore enters BOTH pools.
CODEBOOK_ANCHORS += [
    {"response_id": "R_3sisTwYexKNQbvd", "item_type": "forward",
     "label": "MIXED-MOTIVE (teaching case)",
     "quote": ("there needs to be an appropriate consequence to causing the death "
               "of another human being... An eye for an eye in a sense."),
     "targets": {"proportionality": 0.6, "revenge": 0.6, "moral_balance": 0.4},
     "mixed_motive": True},

    # The same participant, two items: the pair shows the same person reaching for
    # payback on one item and desert on another.
    {"response_id": "R_37e5e0QC8RzXdaV", "item_type": "ranking",
     "label": "MIXED-MOTIVE #2 (ranking item)",
     "quote": "It serves them right to get a taste of payback",
     "targets": {"revenge": 0.7}, "mixed_motive": True},
    {"response_id": "R_37e5e0QC8RzXdaV", "item_type": "backward",
     "label": "MIXED-MOTIVE #2 (backward item)",
     "quote": "He still deserves to go to prison... a miscarriage of justice",
     "targets": {"desert": 0.7, "moral_balance": 0.4}, "mixed_motive": True},
]

# THE HARD NEGATIVE IS RECORDED AND NOT USED. R_7V2vfczhki64TTT [ranking] --
# "Deterrence is shaky, so 'deserving' punishment is just revenge, not a real
# solution." -- is the codebook's stance anchor: the speaker EQUATES desert with
# revenge and REJECTS both, so the codebook scores desert LOW (~0.2) and revenge LOW.
# It teaches the rubric to read stance rather than spot tokens, which is exactly why
# it must not become a reference passage: an encoder has no stance detector, so a
# centroid built with this passage would learn the vocabulary of REJECTING desert as
# the vocabulary of desert.
# HOW THE EXCLUSION ACTUALLY WORKS, in two different ways for the two routes:
#   Route 1 does not reach it because it is a separate variable and NOT a member of
#   CODEBOOK_ANCHORS. The target threshold would not save it if somebody appended it:
#   it carries rehabilitation 0.6, above ANCHOR_TARGET_MIN, so the passage that REJECTS
#   punishment would enter the rehabilitation pool, the one facet it is genuinely high
#   on. The check cell asserts that it reached no pool, which is what catches that edit.
#   Route 2 needs an actual filter, because the word list has no stance detector at all.
#   It scores this response desert 1, revenge 1, deterrence 1 and incapacitation 1, and
#   flags it as both principled and vengeful, so it would be picked up as a candidate
#   for four pools and handed one of the RESERVED mixed-motive slots. HELD_OUT_KEYS is
#   what keeps it out, and it is applied where the candidates are assembled.
HARD_NEGATIVE = {"response_id": "R_7V2vfczhki64TTT", "item_type": "ranking",
                 "label": "HARD-NEGATIVE / stance (not used as a reference)",
                 "quote": ("Deterrence is shaky, so 'deserving' punishment is just "
                           "revenge, not a real solution."),
                 "targets": {"desert": 0.2, "revenge": 0.1, "rehabilitation": 0.6}}
HELD_OUT_KEYS = {(HARD_NEGATIVE["response_id"], HARD_NEGATIVE["item_type"])}

print(f"Codebook anchors on file: {len(CODEBOOK_ANCHORS)} usable, "
      f"{len(HELD_OUT_KEYS)} response held out of every route "
      f"({HARD_NEGATIVE['response_id']} [{HARD_NEGATIVE['item_type']}]).")

Codebook anchors on file: 8 usable, 1 response held out of every route (R_7V2vfczhki64TTT [ranking]).


In [9]:
# === THE FIREWALL: which rows are SEED, which are TEST, and what may be used ===
# WHAT THIS CELL DOES: turns gold_split_assignment.csv into two lookups, checks them,
# and defines the one function every route below asks before it uses a response.
# THE RULE THIS CELL EXISTS TO OBEY (anti-circularity rule 4): never report a
# convergence number for a stream on responses that seeded that stream. Embedding
# reference passages drawn from a response guarantee that the same response will look
# like a match afterwards, so computing embedding-vs-anything agreement on it would
# report a construction as a finding. Leave-one-out at scoring time (notebook 09)
# removes only a response's similarity to ITSELF -- it cannot remove the shared signal
# running through the rest of a stratum -- so the frozen split is mandatory rather
# than tidy.
#
# THE LABELS ARE CHECKED BEFORE THEY ARE TRUSTED. Both key sets are built by matching
# the exact strings "SEED" and "TEST". A file whose gold_split column came back
# lower-cased from a re-export, or carrying a third label from a hand edit, would build
# two EMPTY key sets: every TEST row would then read as admissible, the firewall would
# be off, and the leak check at the end would pass with nothing to check against. So
# the labels are asserted, and both halves are asserted non-empty.
_labels = set(split["gold_split"].astype(str).unique())
assert _labels <= {"SEED", "TEST"}, (
    f"gold_split_assignment.csv carries labels this notebook does not know: "
    f"{sorted(_labels - {'SEED', 'TEST'})}. The firewall matches the exact strings "
    f"SEED and TEST, so any other spelling empties both halves and silently disarms it.")

SEED_KEYS = set(zip(split.loc[split.gold_split == "SEED", "response_id"],
                    split.loc[split.gold_split == "SEED", "item_type"]))
TEST_KEYS = set(zip(split.loc[split.gold_split == "TEST", "response_id"],
                    split.loc[split.gold_split == "TEST", "item_type"]))
assert SEED_KEYS and TEST_KEYS, (
    f"the frozen split holds {len(SEED_KEYS)} SEED and {len(TEST_KEYS)} TEST rows. "
    f"An empty half means notebook 03 wrote a degenerate split; an empty TEST half in "
    f"particular leaves the firewall below nothing to exclude.")

def split_of(key):
    """SEED, TEST, or not_in_gold_set for one (response_id, item_type) pair.

    A row that is in neither half was never drawn into the gold sample at all. It is
    not a TEST row, so using it as a reference passage does not contaminate any
    convergence number -- but the third label is kept distinct in the provenance so a
    reader can see the difference rather than infer it.
    """
    if key in SEED_KEYS:
        return "SEED"
    if key in TEST_KEYS:
        return "TEST"
    return "not_in_gold_set"

def is_admissible(key):
    """May this response x item become a reference passage? Everything but TEST may.

    The long argument for that rule sits in the route 2 cell, where the filter is
    applied to the whole corpus.
    """
    return key not in TEST_KEYS

print(f"SEED rows: {len(SEED_KEYS)}   TEST rows: {len(TEST_KEYS)}   "
      f"corpus rows in neither half: {len(df) - len(SEED_KEYS) - len(TEST_KEYS)}")
print("TEST feeds every gate and convergence number computed against the gold")
print("sample, and nothing in TEST may be used here.")

SEED rows: 154   TEST rows: 154   corpus rows in neither half: 2536
TEST feeds every gate and convergence number computed against the gold
sample, and nothing in TEST may be used here.


In [10]:
# === ROUTE 1: exemplars from the codebook's own anchor passages ===
# WHAT THIS CELL DOES: turns each anchor into an exemplar for every facet the codebook
# scored at ANCHOR_TARGET_MIN or above, dropping any anchor the frozen split put in
# the TEST half.
# WHY THE REFERENCE PASSAGE IS THE CODEBOOK'S QUOTE AND NOT THE PARTICIPANT'S FULL
# RESPONSE: the codebook author read the whole answer and assigned the target to the
# passage printed in the codebook, cutting the rest. Restoring the cut material would
# keep the target while changing the text it describes. It is not a hypothetical: the
# full desert anchor runs 824 characters, opens on locking criminals up and closes on
# "one less person on the street", which is incapacitation and deterrence, and the full
# proportionality anchor opens "Keeping citizens from being victims is the ideal reason
# why punishment is important". Truncating at MAX_CHARS does not help, because in both
# cases the diluting material is at the FRONT.
# WHY THE TARGET THRESHOLD: only targets at 0.5 and above seed a pool. The
# proportionality anchor carries desert 0.2 and the suffering anchor carries
# proportionality 0.3, and a 0.2 target means the facet is barely there -- a trace, not
# an example. The threshold also cuts every 0.4: moral_balance is scored 0.4 on three
# anchors and nowhere higher, so moral_balance takes NOTHING from this route and its
# pool is built from the word list alone. That is a real cost of the threshold and it
# is stated here rather than left to be discovered from the printed counts.
# WHY AN ANCHOR CAN BE DROPPED: the anchors were chosen when the codebook was written,
# before the split was drawn, so one of them can land in the TEST half. If it does, it
# is dropped here and the drop is printed -- the alternative is a reference passage
# that the downstream TEST comparison has already seen.
def _in_corpus(rid, item):
    """Is this anchor a real response in openends_long.csv? One row per key, checked."""
    return bool(len(df[(df["response_id"] == rid) & (df["item_type"] == item)]))

anchor_exemplars = {f: [] for f in FACET_NAMES}
anchors_dropped, anchors_missing = [], []
for a in CODEBOOK_ANCHORS:
    key   = (a["response_id"], a["item_type"])
    where = split_of(key)
    if not is_admissible(key):
        anchors_dropped.append(f"{a['response_id']} [{a['item_type']}] {a['label']}")
        continue
    if not _in_corpus(*key):
        # The codebook quotes this corpus, so an anchor that is not in it means the
        # corpus and the codebook have drifted apart. The quote is still usable as a
        # reference passage, so this is a printed note rather than a stop.
        anchors_missing.append(f"{a['response_id']} [{a['item_type']}]")
    for f, target in a["targets"].items():
        if f not in FACET_NAMES or target < ANCHOR_TARGET_MIN:
            continue
        anchor_exemplars[f].append({
            "response_id": a["response_id"], "item_type": a["item_type"],
            "text": a["quote"][:MAX_CHARS], "route": "codebook_anchor",
            "source": "MEASUREMENT_DESIGN.md section 4 codebook anchor, as quoted there",
            "gold_split": where, "codebook_target": target,
            "codebook_label": a["label"], "random_seed": RANDOM_SEED,
            "mixed_motive": bool(a.get("mixed_motive", False)),
        })

print("Route 1 -- codebook anchors, per facet:")
print("  " + ", ".join(f"{f}={len(anchor_exemplars[f])}" for f in FACET_NAMES))
if anchors_dropped:
    print("  dropped, because the frozen split put them in the TEST half:")
    for line in anchors_dropped:
        print(f"    {line}")
else:
    print("  none dropped: no codebook anchor landed in the TEST half.")
if anchors_missing:
    print(f"  not found in openends_long.csv (quote used as written): {anchors_missing}")

Route 1 -- codebook anchors, per facet:
  desert=2, proportionality=3, censure=1, moral_balance=0, revenge=3, suffering=1, deterrence=0, incapacitation=0, rehabilitation=0
  none dropped: no codebook anchor landed in the TEST half.


In [11]:
# === ROUTE 2, PART 1: join the word-list counts to the text they were scored on ===
# WHAT THIS CELL DOES: puts each response's verbatim text beside its word-list counts,
# checking the join rather than trusting it.
# WHY THE WORD LIST IS AN ADMISSIBLE SOURCE HERE: it uses no language model at all, so
# a reference pool seeded from it leaves the embedding leg independent of the ensemble
# it is later compared against. It is a low-recall, high-precision instrument -- it
# finds few responses, and the ones it finds do contain the facet's language, which is
# exactly the property a reference passage needs.
# WHY THE JOIN IS CHECKED THREE WAYS: facet_dictionary_scores.csv carries the counts
# and openends_long.csv carries the words, and the passages written into the pools come
# from the second while the ranking comes from the first. A join that fanned out, or
# that matched nothing, would still produce a plausible-looking file.
_right = df[KEYS + ["text"]]
# The right-hand frame of the merge must be unique on the join keys or the join
# multiplies rows: one duplicated response in openends_long.csv would give that
# response two chances of being drawn as an exemplar for the same facet.
assert not _right.duplicated(KEYS).any(), \
    "openends_long.csv is not unique on response_id x item_type at the merge."
_n_before = len(dict_df)
cand = dict_df.merge(_right, on=KEYS, how="left")
# The row count cannot grow while the line above holds, so read this as the guard on
# that assumption AT THE POINT OF USE: anything later merged into these frames, or a
# duplicate that slipped past the load cell, shows up here as a changed count.
assert len(cand) == _n_before, (
    f"the text merge changed the row count, {_n_before} -> {len(cand)}. The word-list "
    f"frame has fanned out and every count below would be inflated.")
# The live check on this merge: a left join keeps a dictionary row whether or not the
# corpus file has its key, so a response scored by notebook 01 and absent from notebook
# 00's file would arrive with a null passage and be written into the JSON as one.
assert cand["text"].notna().all(), (
    f"{int(cand['text'].isna().sum())} word-list rows have no text in "
    f"openends_long.csv. The two files disagree on response_id x item_type.")
cand["text"] = cand["text"].astype(str)
print(f"Route 2 -- word-list rows with their text attached: {len(cand)}.")

Route 2 -- word-list rows with their text attached: 2844.


In [12]:
# === ROUTE 2, PART 2: the firewall, applied to the whole corpus ===
# WHAT THIS CELL DOES: removes every response the frozen split put in the TEST half.
# The line below is the firewall in force, and it is the one judgment call in this
# notebook, so the reasoning is set out in full.
#   WHY NOT "SEED ONLY". MEASUREMENT_DESIGN.md section 5 draws reference passages from
#   the SEED half of the gold sample. In the design's build those passages are
#   HUMAN-CODED exemplars, and the only human-coded rows in existence are the gold
#   sample's, so "the SEED half" and "the rows we are allowed to read" name the same
#   set. Under the registered pipeline there are no human codes and the word list
#   scores the whole corpus, so the two stop coinciding.
#   The gold sample is small on purpose -- 250 to 300 rows, one per participant -- so
#   its SEED half is about 154 rows. The word list fires on roughly 1% of responses for
#   proportionality, censure, moral balance and revenge; 1% of 154 rows is one or two
#   exemplars, well below the floor of 8. A SEED-only rule would therefore delete the
#   embedding leg for five of the nine facets, for a reason that has nothing to do with
#   the corpus and everything to do with the size of a sample drawn for another job.
#   WHAT THE RULE ACTUALLY IS. Anti-circularity rule 4 forbids seeding a stream from
#   the rows its convergence is computed on, and every gate and convergence number
#   taken against the gold sample is computed on TEST. A row in neither half was never
#   drawn into the gold sample and never read by a coder, so it carries no coder signal
#   to leak; it is if anything safer than a SEED row, which shares a stratum and a
#   coder with the TEST rows. Route 1 already stands on this -- all eight codebook
#   anchors sit outside the gold sample -- and notebook 09's audit counts "outside the
#   split sample" as a legitimate origin beside SEED.
#   WHAT THE WIDER RULE COSTS, said here rather than left to be discovered: notebook 10
#   drops rows labelled SEED before it computes the ensemble-by-embedding convergence,
#   so exemplars taken from outside the gold sample are not held out there. The
#   provenance block below records how many there are of each kind, and every
#   exemplar's response_id and item_type is written into the file, which is the exact
#   set such a check should hold out.
_n_all = len(cand)
cand = cand[[is_admissible(k) for k in zip(cand["response_id"], cand["item_type"])]].copy()
print(f"  admissible candidates: {len(cand)} of {_n_all} rows "
      f"({_n_all - len(cand)} TEST-half rows removed).")

# The codebook's hard-negative stance passage, removed here and not only from the
# anchors. The word list scores it positive on desert, revenge, deterrence and
# incapacitation and flags it both principled and vengeful, so without this line it
# would arrive as a candidate for four pools with a reserved mixed-motive slot waiting
# for it -- and the reference set for revenge would be built partly from a response
# whose author calls revenge "not a real solution".
_n_adm = len(cand)
cand = cand[[k not in HELD_OUT_KEYS for k in zip(cand["response_id"], cand["item_type"])]].copy()
if len(cand) < _n_adm:
    print(f"  held out of every pool: {_n_adm - len(cand)} row(s) the codebook records "
          f"as stance cases ({sorted(HELD_OUT_KEYS)}).")

  admissible candidates: 2690 of 2844 rows (154 TEST-half rows removed).
  held out of every pool: 1 row(s) the codebook records as stance cases ([('R_7V2vfczhki64TTT', 'ranking')]).


In [13]:
# === ROUTE 2, PART 3: the frozen shuffle, and what the candidate pool holds ===
# Shuffled ONCE with RANDOM_SEED before any sorting below, so that where two candidates
# tie on every ranking key the winner is frozen by the seed rather than decided by the
# order rows happen to sit in the CSV. Re-running the notebook picks the same
# exemplars; re-running it after re-sorting the input picks the same exemplars too.
cand = cand.sample(frac=1.0, random_state=RANDOM_SEED)

if RUN_MODE == "smoke":
    # Smoke mode trims by PARTICIPANT, never by row. A head(20) here would keep 20
    # ROWS, which is six and two thirds of a person: one participant would have two of
    # their three answers inside the run and one outside, and no printed number would
    # say so. Smoke pools are thin by construction -- the relaxed floor in the settings
    # cell lets them through so the rest of the chain has something to run on -- and
    # they are never read as a result.
    _keep = list(pd.unique(cand["response_id"]))[:20]
    cand = cand[cand["response_id"].isin(_keep)].copy()
    print(f"  RUN_MODE=smoke: {len(_keep)} participants, {len(cand)} candidate rows.")

# The composition of the candidate pool, printed because it is the thing a reader would
# otherwise have to take on trust. SEED rows are the half of the gold sample this
# notebook is allowed to read; the rest were never drawn into the gold sample at all.
_by_split = {}
for k in zip(cand["response_id"], cand["item_type"]):
    _by_split[split_of(k)] = _by_split.get(split_of(k), 0) + 1
print("  candidate rows by frozen-split label: "
      + ", ".join(f"{k}={v}" for k, v in sorted(_by_split.items())))

  candidate rows by frozen-split label: SEED=154, not_in_gold_set=2535


In [14]:
# === ROUTE 2, PART 4: how a candidate is ranked ===
# WHAT THIS CELL DOES: defines the per-facet ranking that decides which word-list
# positives become reference passages.
# WHY LOW-OPPOSING FIRST: a cleaner exemplar makes a cleaner reference centroid -- the
# average position in meaning-space of the passages in the set. A response carrying
# both desert and revenge language would pull the desert reference toward revenge and
# blunt the very distinction the embedding leg is asked to measure.
# WHY THE NEGATION-ZEROED COLUMN IS USED AS A FILTER: a whitelisted negation means the
# cue was present and negated -- "I do not want him to suffer" is the clearest possible
# example of what suffering language is NOT, and a centroid built on it would learn the
# opposite of the facet. Note this is a selection filter, never a reported number: the
# negation-adjusted column is barred from every gate computation (anti-circularity
# rule 12), and nothing in this notebook feeds a gate.
def dictionary_candidates(facet):
    """Admissible word-list positives for one facet, best exemplar first."""
    pos = cand[cand[facet] > 0].copy()
    nz = facet + "_negzeroed"
    if nz in pos.columns:
        pos = pos[pos[nz] > 0]
    # Short answers carry no context for an encoder to read.
    if "n_words" in pos.columns:
        pos = pos[pos["n_words"] >= MIN_WORDS]
    opp = [o for o in OPP_OF.get(facet, []) if o in pos.columns]
    pos["_opp"] = pos[opp].fillna(0).sum(axis=1) if opp else 0
    # Three keys, in order: least opposing-facet language, then the strongest showing
    # of this facet, then the shortest answer -- reference passages are short by
    # design, because a long rambling answer dilutes the centroid with everything else
    # the participant happened to say. mergesort is the stable sort, so the frozen
    # shuffle above still decides exact ties.
    keys = ["_opp", facet] + (["n_words"] if "n_words" in pos.columns else [])
    return pos.sort_values(keys, ascending=[True, False] + ([True] if len(keys) == 3 else []),
                           kind="mergesort")

def dict_exemplar(row, facet):
    """One candidate row, as an exemplar record carrying its own provenance."""
    key = (row["response_id"], row["item_type"])
    return {
        "response_id": row["response_id"], "item_type": row["item_type"],
        # Truncated because these are REFERENCE PASSAGES: short by design.
        "text": str(row["text"])[:MAX_CHARS],
        "route": "wordlist_non_test",
        "source": ("facet_dictionary_scores.csv, outside the TEST half of "
                   "gold_split_assignment.csv"),
        # Looked up rather than written in as a constant. A label typed here would
        # simply repeat this notebook's intention back to whoever reads the file, and
        # would keep saying SEED however the exemplar was actually obtained.
        "gold_split": split_of(key), "random_seed": RANDOM_SEED,
        "dict_hits": int(row[facet]),
        "opposing_hits": int(row["_opp"]),
        "mixed_motive": bool(row.get("principled_any", 0) and row.get("vengeful_any", 0)),
    }

In [16]:
# === ASSEMBLE THE POOLS: anchors first, mixed motives reserved, word list to fill ===
# WHAT THIS CELL DOES: fills each facet's pool to at most POOL_MAX exemplars, in a
# fixed order of preference, with no response used twice in the same pool.
# THE ORDER, AND WHY: the codebook anchors go in first because they are the study's own
# written definition of the facet. The word list fills whatever is left.
# THE RESERVED MIXED-MOTIVE SLOTS: the low-opposing ranking above actively pushes
# responses that carry BOTH principled and vengeful language to the bottom of the
# queue, which, left alone, would build pools out of pure cases only and teach the
# embedding leg that the two registers never co-occur -- the exact proposition E4 is
# meant to test. So MIXED_MIN slots are held for genuine mixed-motive responses before
# the pool is filled with clean ones, and a mixed-motive exemplar goes into BOTH the
# principled and the vengeful pools it belongs to.
def _add(picked, seen, exemplar):
    """Append an exemplar unless the pool is full or already holds that response."""
    key = (exemplar["response_id"], exemplar["item_type"])
    if key in seen or len(picked) >= POOL_MAX:
        return
    seen.add(key)
    picked.append(exemplar)

pools = {}
for f in FACET_NAMES:
    picked, seen = [], set()
    for e in anchor_exemplars[f]:
        _add(picked, seen, e)
    ranked = dictionary_candidates(f)
    if len(ranked):
        # (1) the reserved slots. The two any-hit flags come from notebook 01; if a
        # hand-edited file arrives without them there is simply no reserve to fill.
        has_flags = {"principled_any", "vengeful_any"}.issubset(ranked.columns)
        mixed = (ranked[(ranked["principled_any"] > 0) & (ranked["vengeful_any"] > 0)]
                 if has_flags else ranked.iloc[0:0])
        for _, r in mixed.head(MIXED_MIN).iterrows():
            _add(picked, seen, dict_exemplar(r, f))
        # (2) the clean fill
        for _, r in ranked.iterrows():
            _add(picked, seen, dict_exemplar(r, f))
    pools[f] = picked

print("Assembled pools (before the minimum-size check):")
for f in FACET_NAMES:
    routes = {}
    for e in pools[f]:
        routes[e["route"]] = routes.get(e["route"], 0) + 1
    detail = ", ".join(f"{k}={v}" for k, v in sorted(routes.items())) or "none"
    print(f"  {f:16s} {len(pools[f]):2d}  ({detail})")

Assembled pools (before the minimum-size check):
  desert           15  (codebook_anchor=2, wordlist_non_test=13)
  proportionality  15  (codebook_anchor=3, wordlist_non_test=12)
  censure          15  (codebook_anchor=1, wordlist_non_test=14)
  moral_balance    14  (wordlist_non_test=14)
  revenge          14  (codebook_anchor=3, wordlist_non_test=11)
  suffering        15  (codebook_anchor=1, wordlist_non_test=14)
  deterrence       15  (wordlist_non_test=15)
  incapacitation   15  (wordlist_non_test=15)
  rehabilitation   15  (wordlist_non_test=15)


In [17]:
# === THE MINIMUM-SIZE CHECK: a thin pool is emptied and reported, never used ===
# WHAT THIS CELL DOES: empties any pool holding fewer than POOL_MIN exemplars, records
# why, and says out loud how many facets are left without an embedding measurement.
# WHY EMPTY RATHER THAN THIN: the design sets 8 to 15 exemplars per facet because a
# mean cosine -- the average closeness of a response to the passages in the set -- is
# far less sensitive to any one passage when it is taken over that many. A centroid
# built from two or three responses is one unusual response away from meaning something
# else, and the similarity column it produces would look like a measurement while being
# noise. An empty pool is legible downstream: the facet simply carries no embedding
# measurement, and the notebook that reports convergence says "not run" for it.
# WHAT A SHORTFALL MEANS: the admissible corpus does not contain the facet in language
# the word list recognises. The list is high-precision and low-recall by construction
# and fires on roughly 1% of responses for proportionality, censure, moral balance and
# revenge, so those four are where a shortfall would show first. What is lost is the
# facet's similarity column in notebook 09 and its ensemble-by-embedding convergence
# line in notebook 10. What is NOT affected is the registered gate: both its clauses
# (inter-model ICC >= .60, and r >= .30 with the matching survey scale) are computed
# without any embedding.
pool_status = {}
for f in FACET_NAMES:
    n = len(pools[f])
    if n >= POOL_MIN:
        pool_status[f] = {"status": "ok", "n_exemplars": n, "n_found": n}
        continue
    pool_status[f] = {
        "status": "empty_below_minimum", "n_exemplars": 0, "n_found": n,
        "why": (f"only {n} exemplar(s) available from the codebook anchors plus the "
                f"admissible word-list positives, below the minimum of {POOL_MIN}"),
    }
    pools[f] = []
    print(f"  WARNING  {f}: {n} exemplar(s) found, below the minimum of {POOL_MIN}.")
    print(f"           Pool written empty. This facet carries no embedding")
    print(f"           measurement, which is reported as 'not run', not as a failure.")

# THE NAMED ALARM. Without it, a run that writes nine full pools and a run that writes
# nine empty ones take the same exit path -- some prints, done(), "done." -- and the
# difference between a measurement and an empty file is left for a reader to notice in
# a table. This block is what makes the useless run announce itself.
_short = [f for f in FACET_NAMES if pool_status[f]["status"] != "ok"]
if not _short:
    print(f"  All {len(FACET_NAMES)} facets reached the minimum pool size.")
else:
    print("  " + "!" * 68)
    print(f"  {len(_short)} of {len(FACET_NAMES)} facets have NO reference pool: "
          + ", ".join(_short))
    print("  Those facets carry no embedding measurement anywhere downstream. If E4's")
    print("  pair (desert, revenge) or a facet you mean to report is on that list, the")
    print("  embedding leg is not available for it and the convergence table will say")
    print("  so. Do not read an empty pool as a null result.")
    print("  " + "!" * 68)

  All 9 facets reached the minimum pool size.


In [18]:
# === THE HUMAN-CODED ARM: a comparison, and never an input to the pools ===
# WHAT THIS CELL DOES: when HUMAN_GOLD is True and gold_codes.csv is present and
# readable, it compares the pools that were just built against what human coders marked
# positive. It READS the pools; it never adds to them.
# WHY IT MUST NOT FEED THE POOLS, even though the design would rather have human
# exemplars: prereg 6.7 prescribes no human-coding step, so the human arm is a
# deviation confined to notebook 99, and nothing in notebooks 00 to 12 may depend on
# whether it was run. Human codes added here would change embedding_seed_pools.json,
# and with it every vector notebook 08 embeds, every similarity notebook 09 computes,
# and every convergence line notebooks 10 and 12 print -- so the registered pipeline
# would produce two different sets of numbers depending on an unregistered arm. The
# pools written below are therefore identical whichever way HUMAN_GOLD is set, and the
# human codes are reported here as what they are: an independent opinion on whether the
# word list picked passages a coder would also call positive.
# NO HARD STOP: a defect in this optional file is printed and survived. It cannot take
# down a notebook whose registered output does not use it.
human_check = {"status": "not run",
               "why": "HUMAN_GOLD is False, the registered pipeline (prereg 6.7)"}
if HUMAN_GOLD and gold is None:
    human_check["why"] = "HUMAN_GOLD is True but gold_codes.csv is not in the folder"
elif HUMAN_GOLD and gold_problem:
    human_check["why"] = gold_problem
elif gold is not None:
    _fac = [f for f in FACET_NAMES if f in gold.columns]
    if not _fac:
        human_check["why"] = "gold_codes.csv carries none of the nine facet columns"
    else:
        _coded = set(zip(gold["response_id"], gold["item_type"]))
        _rows = {}
        for f in _fac:
            _pos = gold[pd.to_numeric(gold[f], errors="coerce").fillna(0) > 0]
            _pos_keys = {k for k in zip(_pos["response_id"], _pos["item_type"])}
            _written = {(e["response_id"], e["item_type"]) for e in pools[f]}
            # Only the exemplars a coder actually read can be compared at all. Most
            # exemplars sit outside the gold sample, so this number is small by
            # construction and a low overlap is not a disagreement.
            _checkable = _written & _coded
            _rows[f] = {"human_positives_admissible":
                            len([k for k in _pos_keys if is_admissible(k)]),
                        "pool_exemplars_a_coder_read": len(_checkable),
                        "of_those_coded_positive": len(_checkable & _pos_keys)}
        human_check = {"status": "computed", "per_facet": _rows, "why": ""}

if human_check["status"] != "computed":
    print(f"Human-coded comparison: NOT RUN -- {human_check['why']}.")
else:
    print("Human-coded comparison (the pools are unchanged by it):")
    print(f"  {'facet':16s} {'human pos':>9s} {'in pools':>9s} {'agreed':>7s}")
    for f, r in human_check["per_facet"].items():
        print(f"  {f:16s} {r['human_positives_admissible']:9d} "
              f"{r['pool_exemplars_a_coder_read']:9d} {r['of_those_coded_positive']:7d}")

Human-coded comparison: NOT RUN -- HUMAN_GOLD is False, the registered pipeline (prereg 6.7).


In [19]:
# === THE GUARANTEES, CHECKED RATHER THAN ASSERTED IN PROSE ===
# WHAT THIS CELL DOES: walks every exemplar that is about to be written and checks
# three properties that no comment can be trusted to have arranged.
#   (1) NO TEST-HALF RESPONSE. This is the one property the whole anti-circularity
#       argument rests on. If a TEST row reached a reference pool, the convergence
#       numbers computed downstream would be partly self-fulfilling and nothing in the
#       output would show it.
#   (2) THE HARD NEGATIVE REACHED NO POOL. HELD_OUT_KEYS keeps it out of the word-list
#       route, but nothing keeps it out of route 1 except its absence from
#       CODEBOOK_ANCHORS, and appending it there is the obvious tidy-up. This line is
#       what makes the exclusion survive that edit.
#   (3) NO POOL HOLDS THE SAME RESPONSE TWICE. A response counted twice weights itself
#       twice in the facet's centroid, which is the same defect as a duplicated row in
#       an input file and just as invisible.
# The counts are taken from split_of(), not from the gold_split label written beside
# each exemplar, because a label checked against the place it was copied from proves
# nothing. They run AFTER the minimum-size cell emptied the thin pools, so what is
# inspected here is exactly what will be written to disk.
pool_split_counts = {"SEED": 0, "TEST": 0, "not_in_gold_set": 0}
_leaked = []
for f in FACET_NAMES:
    for e in pools[f]:
        _lab = split_of((e["response_id"], e["item_type"]))
        pool_split_counts[_lab] += 1
        if _lab == "TEST":
            _leaked.append((e["response_id"], e["item_type"], f))
assert not _leaked, ("TEST-half responses reached the reference pools: "
                     f"{_leaked[:5]} -- the SEED/TEST firewall is broken.")

_hard_key = (HARD_NEGATIVE["response_id"], HARD_NEGATIVE["item_type"])
_hard_in = [f for f in FACET_NAMES for e in pools[f]
            if (e["response_id"], e["item_type"]) == _hard_key]
assert not _hard_in, (
    f"the hard-negative stance passage {_hard_key} reached the pools for {_hard_in}. "
    f"That response REJECTS punishment, so a centroid built with it would learn the "
    f"vocabulary of rejecting a facet as the vocabulary of the facet.")

for f in FACET_NAMES:
    _ks = [(e["response_id"], e["item_type"]) for e in pools[f]]
    assert len(_ks) == len(set(_ks)), (
        f"the {f} pool holds the same response x item more than once: "
        f"{[k for k in _ks if _ks.count(k) > 1][:3]}. A repeated exemplar weights "
        f"itself twice in that facet's centroid.")

_n_ex   = sum(len(pools[f]) for f in FACET_NAMES)
_n_rows = len({(e["response_id"], e["item_type"]) for f in FACET_NAMES for e in pools[f]})
print(f"{_n_ex} exemplars across {len(FACET_NAMES)} pools, drawn from {_n_rows} "
      f"distinct responses, none of them used twice within a pool.")
print(f"  from the SEED half        : {pool_split_counts['SEED']}")
print(f"  from outside the gold set : {pool_split_counts['not_in_gold_set']}")
print(f"  from the TEST half        : {pool_split_counts['TEST']}   (counted here, "
      f"not claimed)")

133 exemplars across 9 pools, drawn from 124 distinct responses, none of them used twice within a pool.
  from the SEED half        : 3
  from outside the gold set : 130
  from the TEST half        : 0   (counted here, not claimed)


In [20]:
# === THE PROVENANCE BLOCK: what a reader needs in order to audit the pools ===
# WHAT THIS CELL DOES: assembles the object that gets written to disk -- the pools
# themselves plus a record of how they were built.
# WHY THE PROVENANCE TRAVELS INSIDE THE FILE: the notebook that reports embedding
# convergence prints this block, so it can state where its reference passages came from
# without trusting a filename.
# EVERY CLAIM IN IT IS COMPUTED, NOT TYPED. The source sentence is built from the routes
# that actually put an exemplar in a pool, and the TEST-row count is the number counted
# in the cell above. A hardcoded zero there would read as a guarantee while being a
# statement of intent: it would say zero just as loudly on a run that leaked.
_routes_used = sorted({e["route"] for f in FACET_NAMES for e in pools[f]})
_route_words = {
    "codebook_anchor": "codebook anchor passages (MEASUREMENT_DESIGN.md section 4)",
    "wordlist_non_test": ("word-list-positive responses from outside the TEST half of "
                          "the frozen split"),
}
_source = (" plus ".join(_route_words.get(r, r) for r in _routes_used)
           or "no exemplars reached any pool")
# Counted, not typed as a zero. If the human arm were ever wired into the assembly
# cell, this number would move and the sentence below would report it.
_n_human = sum(1 for f in FACET_NAMES for e in pools[f] if "human" in e["route"])

seed_pools_out = {
    # The two fields the downstream loader reads by name.
    "random_seed": RANDOM_SEED,
    "source": (f"{_source}; no ensemble output was consulted; TEST-half responses "
               f"used: {pool_split_counts['TEST']}; human-coded exemplars used: "
               f"{_n_human}"),
    # The run-level record.
    "built_by": f"{NB_ID}_{NB_NAME}",
    "built_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "run_mode": RUN_MODE,
    "human_gold": HUMAN_GOLD,
    "routes": {
        "codebook_anchor": ("author-written anchor passages from the codebook, quoted "
                            "as the codebook prints them, independent of the ensemble "
                            "by construction"),
        "wordlist_non_test": ("responses the nine-facet word list scored positive, "
                              "ranked low on the opposing facets, drawn from "
                              "everywhere except the TEST half"),
    },
    "routes_that_placed_an_exemplar": _routes_used,
    "settings": {"pool_min": POOL_MIN, "pool_max": POOL_MAX,
                 "mixed_motive_reserved": MIXED_MIN, "max_chars": MAX_CHARS,
                 "min_words": MIN_WORDS, "anchor_target_min": ANCHOR_TARGET_MIN},
    "split": {"seed_rows": len(SEED_KEYS), "test_rows": len(TEST_KEYS),
              "test_rows_used": pool_split_counts["TEST"],
              "seed_rows_used": pool_split_counts["SEED"],
              "outside_gold_sample_rows_used": pool_split_counts["not_in_gold_set"],
              "codebook_anchors_dropped_as_TEST": anchors_dropped},
    "convergence_holdout": (
        "a convergence number computed against this embedding leg should hold out the "
        "response_id x item_type pairs listed under pools: those are the rows the leg "
        "was seeded from. Notebook 10 holds out rows labelled SEED in "
        f"gold_split_assignment.csv, which covers {pool_split_counts['SEED']} of them "
        f"and not the {pool_split_counts['not_in_gold_set']} drawn from outside the "
        "gold sample."),
    "human_gold_cross_check": human_check,
    "hard_negative_excluded": HARD_NEGATIVE,
    "facet_status": pool_status,
    "pools": pools,
}
print(f"Provenance block assembled: {len(seed_pools_out['pools'])} facets, "
      f"routes that placed an exemplar = {_routes_used}.")

Provenance block assembled: 9 facets, routes that placed an exemplar = ['codebook_anchor', 'wordlist_non_test'].


## Save and check

In [21]:
# === SAVE: write embedding_seed_pools.json and record it in the manifest ===
# WHAT THIS CELL DOES: writes the one output of this notebook and registers it with
# done(), so the notebooks downstream can find it and check it has not changed.
# THE LAST CONTRACT CHECK, on the object rather than on a file it came from: no facet's
# pool may hold the same response x item twice, and every exemplar must carry the two
# keys the whole chain joins on. Notebook 08 reads item_type with .get() and would turn
# a missing one into a passage that matches no response and quietly leaves every
# reference set, so it is caught here instead.
for f, ex in pools.items():
    _ks = [(e.get("response_id"), e.get("item_type")) for e in ex]
    assert all(k[0] and k[1] for k in _ks), \
        f"an exemplar in the {f} pool is missing response_id or item_type: {_ks}"
    assert len(_ks) == len(set(_ks)), f"the {f} pool repeats a response x item pair"

OUT = "embedding_seed_pools.json"
with open(OUT, "w") as fh:
    json.dump(seed_pools_out, fh, indent=2)
# The note is built from what happened, not from what was meant to happen: the counts
# come from the check cell, so a run that leaked would say so here too.
done(OUT, rows=sum(len(v) for v in pools.values()),
     note=(f"{POOL_MIN}-{POOL_MAX} reference passages per facet; "
           f"{sum(1 for f in FACET_NAMES if pool_status[f]['status'] == 'ok')} of "
           f"{len(FACET_NAMES)} facets filled; routes {_routes_used}; "
           f"TEST-half rows used: {pool_split_counts['TEST']}"))

  [wrote] embedding_seed_pools.json  (133 rows, 80,542 bytes)


In [22]:
# === CHECK: the per-facet table a reader should look at ===
# WHAT TO READ HERE: one line per facet, with the number of reference passages, where
# they came from and the first few words of the first one. The pattern to expect is
# that the sparse facets -- revenge, moral balance, proportionality, censure -- carry
# the fewest word-list exemplars, because the list fires on about 1% of responses for
# each of them, and that the three consequentialist goals are the full ones. A facet
# reading "empty" carries no embedding measurement at all, which is reported downstream
# as "not run", never as a failed check.
print("FACET             n  routes                                  example")
print("-" * 78)
for f in FACET_NAMES:
    ex = pools[f]
    if not ex:
        print(f"{f:16s}  0  (empty: {pool_status[f]['n_found']} found, "
              f"minimum {POOL_MIN})")
        continue
    routes = {}
    for e in ex:
        routes[e["route"]] = routes.get(e["route"], 0) + 1
    rtxt = ", ".join(f"{k.split('_')[0]}:{v}" for k, v in sorted(routes.items()))
    snippet = ex[0]["text"].replace("\n", " ")[:28]
    print(f"{f:16s} {len(ex):2d}  {rtxt:38s} {snippet}...")

# The mixed-motive line, counted rather than asserted. What matters is not how many
# mixed-motive passages were carried in but how many of them sit in more than one
# facet's pool: a mixed-motive exemplar that lands in only one pool teaches the encoder
# nothing about co-occurrence, which is the whole reason the slots are reserved.
_mixed_keys = {(e["response_id"], e["item_type"])
               for f in FACET_NAMES for e in pools[f] if e.get("mixed_motive")}
_pools_of = {k: sum(1 for f in FACET_NAMES for e in pools[f]
                    if (e["response_id"], e["item_type"]) == k) for k in _mixed_keys}
_in_two = sum(1 for k, n in _pools_of.items() if n > 1)
print(f"\nMixed-motive responses carried into the pools: {len(_mixed_keys)}, "
      f"of which {_in_two} appear in more than one facet's pool.")
if _in_two:
    print("Those are the passages that keep the reference sets from teaching the")
    print("encoder that principled and vengeful language never co-occur, which is the")
    print("exact proposition E4 exists to test.")
else:
    print("None of them appears in two pools, so the reference sets carry no evidence")
    print("of co-occurrence. E4's question is not settled by that -- it is measured on")
    print("the ensemble scores -- but the embedding leg has nothing to say about it.")

FACET             n  routes                                  example
------------------------------------------------------------------------------
desert           15  codebook:2, wordlist:13                Criminals deserve to be puni...
proportionality  15  codebook:3, wordlist:12                They should pay a reasonable...
censure          15  codebook:1, wordlist:14                The sentencing should always...
moral_balance    14  wordlist:14                            debt to society and am messa...
revenge          14  codebook:3, wordlist:11                eye for an eye is what i bel...
suffering        15  codebook:1, wordlist:14                since he's paralyzed, he'll ...
deterrence       15  wordlist:15                            I selected that criminals de...
incapacitation   15  wordlist:15                            It will keep him off the str...
rehabilitation   15  wordlist:15                            criminals deserve to be puni...

Mixed-motive responses 

In [23]:
# === WHAT THIS NOTEBOOK PRODUCED, AND WHAT RUNS NEXT ===
print(f"NOTEBOOK {NB_ID} -- {NB_NAME}: done.")
print()
print("  Wrote embedding_seed_pools.json: the reference passages each facet's")
print("  embedding similarity is measured against, with the provenance of every one.")
print()
print("  How the pools were built, in one paragraph:")
print("    Each facet's passages come from the codebook's own written anchors and")
print("    from responses the word list scored positive for that facet. Neither route")
print("    consults a language model, so the embedding leg stays an independent check")
print("    on the ensemble. The TEST half is held out, which is what lets the")
print("    convergence numbers computed on TEST mean something, and the number of")
print(f"    TEST-half rows this run actually used is {pool_split_counts['TEST']}, "
      f"counted rather than")
print("    promised. Facets whose pools came up short are written empty and reported")
print("    as 'not run' rather than left to look like a failed check.")
print()
print(f"  Pools filled: "
      f"{sum(1 for f in FACET_NAMES if pool_status[f]['status'] == 'ok')} of "
      f"{len(FACET_NAMES)} facets. Exemplars by origin: "
      f"SEED {pool_split_counts['SEED']}, outside the gold sample "
      f"{pool_split_counts['not_in_gold_set']}, TEST {pool_split_counts['TEST']}.")
print()
print("  The SEED half has two other jobs in the design (section 8): the data-driven")
print("  word-list expansion, and the queue of negated cues to inspect by hand. Both")
print("  need human codes to exist at all, so under the registered pipeline neither is")
print("  produced and the word list ships exactly as notebook 01 wrote it. They belong")
print("  to the optional human-coding notebook, not to this one.")
print()
print("  Next: the embedding notebook reads this file and embeds these passages in the")
print("  same space as the corpus. Notebook 09 prints the provenance recorded here, so")
print("  the pools can be audited without opening the JSON.")

NOTEBOOK 04 -- reference_pools: done.

  Wrote embedding_seed_pools.json: the reference passages each facet's
  embedding similarity is measured against, with the provenance of every one.

  How the pools were built, in one paragraph:
    Each facet's passages come from the codebook's own written anchors and
    from responses the word list scored positive for that facet. Neither route
    consults a language model, so the embedding leg stays an independent check
    on the ensemble. The TEST half is held out, which is what lets the
    convergence numbers computed on TEST mean something, and the number of
    TEST-half rows this run actually used is 0, counted rather than
    promised. Facets whose pools came up short are written empty and reported
    as 'not run' rather than left to look like a failed check.

  Pools filled: 9 of 9 facets. Exemplars by origin: SEED 3, outside the gold sample 130, TEST 0.

  The SEED half has two other jobs in the design (section 8): the data-driven
